# Qwen3.5 + SGLang GPU Demo

この Notebook は、SGLang で Qwen3.5 を GPU 上で起動し、テキスト入力と画像入力の両方を試すデモです。複数のサンプル画像・サンプルプロンプトも含みます。


## 1. 前提
- Docker が使える
- NVIDIA GPU が見えている
- Hugging Face のトークンが必要な場合は `HF_TOKEN` を設定する
- Blackwell 系 GPU では `--attention-backend triton` が必要


In [ ]:
!nvidia-smi


In [ ]:
!python scripts/generate_sample_assets.py


## 2. SGLang サーバ起動
下のセルは Docker コンテナをバックグラウンド起動します。必要に応じてモデル名を変更してください。


In [ ]:
from pathlib import Path
import os, shlex, subprocess, time, requests

MODEL = os.environ.get('MODEL_NAME', 'Qwen/Qwen3.5-4B')
HF_TOKEN = os.environ.get('HF_TOKEN', '')
PROJECT = Path.cwd()
subprocess.run('docker rm -f qwen35-sglang-demo >/dev/null 2>&1 || true', shell=True, check=False)
cmd = f'''docker run -d --rm --name qwen35-sglang-demo --gpus all --ipc=host --shm-size 16g -p 30000:30000 -v {PROJECT / 'assets'}:/workspace/assets -v {Path.home() / '.cache' / 'huggingface'}:/root/.cache/huggingface -e HF_TOKEN={shlex.quote(HF_TOKEN)} lmsysorg/sglang:latest-cu130-runtime python3 -m sglang.launch_server --model-path {shlex.quote(MODEL)} --host 0.0.0.0 --port 30000 --tp-size 1 --mem-fraction-static 0.8 --context-length 32768 --attention-backend triton --reasoning-parser qwen3'''
print(cmd)
subprocess.run(cmd, shell=True, check=True)
for _ in range(180):
    try:
        r = requests.get('http://127.0.0.1:30000/v1/models', timeout=5)
        if r.ok:
            print('Server is ready')
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError('Server did not become ready in time')


In [ ]:
!docker logs qwen35-sglang-demo --tail 120


## 3. 単発のテキスト入力テスト


In [ ]:
!python scripts/text_request.py --model Qwen/Qwen3.5-4B


## 4. 単発の画像入力テスト


In [ ]:
!python scripts/vision_request.py --model Qwen/Qwen3.5-4B --image assets/sample_shapes.png


## 5. サンプルスイート実行
複数のテキストプロンプトと画像サンプルを一括実行して JSON 保存します。


In [ ]:
import base64, json, mimetypes, requests
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / 'notebooks').exists():
    PROJECT = ROOT
elif ROOT.name == 'notebooks':
    PROJECT = ROOT.parent
else:
    PROJECT = ROOT

MODEL = 'Qwen/Qwen3.5-4B'
API_BASE = 'http://127.0.0.1:30000/v1'
OUT = PROJECT / 'outputs' / 'sample_suite_results.json'
OUT.parent.mkdir(parents=True, exist_ok=True)

def image_to_data_url(path: Path) -> str:
    mime = mimetypes.guess_type(path.name)[0] or 'image/png'
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'

def chat(messages, max_tokens=256):
    payload = {'model': MODEL, 'messages': messages, 'max_tokens': max_tokens}
    r = requests.post(f'{API_BASE}/chat/completions', json=payload, timeout=900)
    r.raise_for_status()
    return r.json()

text_prompts = [
    'Explain in 3 bullet points what SGLang is and why it is useful.',
    'Summarize how to serve Qwen3.5 with an OpenAI-compatible API in 4 short bullet points.',
    'Give a concise explanation of why GPU inference frameworks matter for multimodal models.',
]
vision_cases = [
    (PROJECT / 'assets' / 'sample_shapes.png', 'Describe the image in detail. Count the shapes and mention their colors.'),
    (PROJECT / 'assets' / 'sample_chart.png', 'Read this chart. Which quarter is highest, and list the bars in descending order.'),
    (PROJECT / 'assets' / 'sample_receipt.png', 'Extract the key information from this receipt, including store, date, and total.'),
]
results = {'text': [], 'vision': []}
for prompt in text_prompts:
    results['text'].append({'prompt': prompt, 'response': chat([{'role': 'user', 'content': prompt}])})
for image_path, prompt in vision_cases:
    p = Path(image_path)
    results['vision'].append({
        'image': p.name,
        'prompt': prompt,
        'response': chat([{'role': 'user', 'content': [
            {'type': 'text', 'text': prompt},
            {'type': 'image_url', 'image_url': {'url': image_to_data_url(p)}},
        ]}])
    })
OUT.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print(OUT)


In [ ]:
import json
from pathlib import Path
ROOT = Path.cwd()
if (ROOT / 'notebooks').exists():
    PROJECT = ROOT
elif ROOT.name == 'notebooks':
    PROJECT = ROOT.parent
else:
    PROJECT = ROOT
out = PROJECT / 'outputs' / 'sample_suite_results.json'
result = json.loads(out.read_text(encoding='utf-8'))
print('text cases:', len(result['text']))
print('vision cases:', len(result['vision']))
print('first vision image:', result['vision'][0]['image'])
print(json.dumps(result['vision'][0]['response']['choices'][0]['message'], ensure_ascii=False, indent=2)[:1200])


## 6. Notebook を保存
実行済み Notebook を HTML へも変換できます。


In [ ]:
!jupyter nbconvert --to html notebooks/qwen35_sglang_demo.ipynb --output qwen35_sglang_demo_rendered.html


## 7. 後始末


In [ ]:
!docker rm -f qwen35-sglang-demo || true
